# 01 — Setup and Load Raw Data

Creates the catalog, schema, and volume; copies the CSV files from the workspace into the volume; and loads 3 Delta tables.

In [ ]:
dbutils.widgets.text("catalog",     "medtech",  "Catalog")
dbutils.widgets.text("schema",      "sales",    "Schema")
dbutils.widgets.text("volume_name", "raw_data", "Volume Name")

catalog     = dbutils.widgets.get("catalog")
schema      = dbutils.widgets.get("schema")
volume_name = dbutils.widgets.get("volume_name")

print("catalog:",     catalog)
print("schema:",      schema)
print("volume_name:", volume_name)

## Step 1 — Create Catalog, Schema, and Volume

In [ ]:
%sql
CREATE CATALOG IF NOT EXISTS IDENTIFIER(:catalog);
CREATE SCHEMA  IF NOT EXISTS IDENTIFIER(:catalog || '.' || :schema);
CREATE VOLUME  IF NOT EXISTS IDENTIFIER(:catalog || '.' || :schema || '.' || :volume_name);

## Step 2 — Copy CSVs into Volume

In [ ]:
import os, shutil

notebook_dir  = os.path.dirname(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
raw_data_path = f"/Workspace{notebook_dir}/../raw_data/med_tech_sales"
volume_path   = f"/Volumes/{catalog}/{schema}/{volume_name}"

for csv in ["account_targeting.csv", "hcp_procedure_volume.csv", "product_upgrades.csv"]:
    src = f"{raw_data_path}/{csv}"
    dst = f"{volume_path}/{csv}"
    shutil.copy2(src, dst)
    print(f"Copied {csv}")

print(f"\nFiles in volume:")
for f in dbutils.fs.ls(f"dbfs:{volume_path}"):
    print(f"  {f.name}  ({f.size:,} bytes)")

## Step 3 — Create Delta Tables

In [ ]:
%sql
USE CATALOG IDENTIFIER(:catalog);
USE SCHEMA IDENTIFIER(:schema);

### HCP Procedure Volume (150 rows)

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.hcp_procedure_volume AS
SELECT
  `NPI` AS npi,
  `Surgeon Name` AS surgeon_name,
  `Rep ID` AS rep_id,
  `Account` AS account,
  `Specialty` AS specialty,
  `Area` AS area,
  CAST(`CY Procedure Volume` AS INT) AS cy_procedure_volume,
  CAST(`PY Procedure Volume` AS INT) AS py_procedure_volume,
  CAST(`CY vs PY Procedure Volume` AS INT) AS cy_vs_py_procedure_volume,
  CAST(`CY Alpha Max Market` AS DOUBLE) AS cy_alpha_max_market,
  CAST(`CY Beta Max Market` AS DOUBLE) AS cy_beta_max_market,
  CAST(`CY Gamma Max Market` AS DOUBLE) AS cy_gamma_max_market,
  CAST(`CY Delta Max Market` AS DOUBLE) AS cy_delta_max_market,
  CAST(`Total CY Max Market` AS DOUBLE) AS total_cy_max_market,
  CAST(`PY Alpha Max Market` AS DOUBLE) AS py_alpha_max_market,
  CAST(`PY Beta Max Market` AS DOUBLE) AS py_beta_max_market,
  CAST(`PY Gamma Max Market` AS DOUBLE) AS py_gamma_max_market,
  CAST(`PY Delta Max Market` AS DOUBLE) AS py_delta_max_market,
  CAST(`Total PY Max Market` AS DOUBLE) AS total_py_max_market
FROM read_files(
  '/Volumes/{catalog}/{schema}/{volume_name}/hcp_procedure_volume.csv',
  format => 'csv', header => true, inferSchema => true
)
""")
print("hcp_procedure_volume created")

### Product Upgrades (400 rows)

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.product_upgrades AS
SELECT
  CAST(`Row Number` AS INT) AS row_number,
  `Area` AS area, `Region` AS region, `Territory` AS territory,
  `IDN` AS idn, `Rep ID` AS rep_id, `Account` AS account,
  `Product Line` AS product_line, `Product Category` AS product_category,
  `Product Description` AS product_description, `Comped Status` AS comped_status,
  `Segment` AS segment,
  CAST(`# of Upgrade Codes` AS INT) AS num_upgrade_codes,
  CAST(`# of Codes on Account Shelf` AS INT) AS num_codes_on_account_shelf,
  CAST(`# of Codes in IDN System` AS INT) AS num_codes_in_idn_system,
  CAST(`# of Codes in Neither` AS INT) AS num_codes_in_neither,
  `Discontinuation Date` AS discontinuation_date,
  CAST(`CY Opportunity ($)` AS DOUBLE) AS cy_opportunity,
  CAST(`PY Opportunity ($)` AS DOUBLE) AS py_opportunity,
  CAST(`Net Cost To Customer ($)` AS DOUBLE) AS net_cost_to_customer,
  CAST(`Rolling 12 Sales ($)` AS DOUBLE) AS rolling_12_sales
FROM read_files(
  '/Volumes/{catalog}/{schema}/{volume_name}/product_upgrades.csv',
  format => 'csv', header => true, inferSchema => true
)
""")
print("product_upgrades created")

### Account Targeting (300 rows)

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.account_targeting AS
SELECT
  CAST(`Row Number` AS INT) AS row_number,
  `Area` AS area, `Region` AS region, `Territory` AS territory,
  `Rep ID` AS rep_id, `IDN` AS idn, `Account` AS account,
  `Clinical Focus Area` AS clinical_focus_area, `Target Type` AS target_type,
  `Product Line` AS product_line, `Category` AS category, `GPO` AS gpo,
  CAST(`Total Units Sold` AS INT) AS total_units_sold,
  CAST(`Opportunity ($)` AS DOUBLE) AS opportunity,
  CAST(`Net Cost To Customer ($)` AS DOUBLE) AS net_cost_to_customer,
  CAST(`Rolling 12 Sales ($)` AS DOUBLE) AS rolling_12_sales,
  CAST(`# Codes on Account Shelf` AS INT) AS num_codes_on_account_shelf,
  CAST(`# Codes in System` AS INT) AS num_codes_in_system,
  CAST(`# Codes in Neither` AS INT) AS num_codes_in_neither,
  CAST(`2025 Penetration %` AS DOUBLE) AS penetration_2025,
  CAST(`2024 Penetration %` AS DOUBLE) AS penetration_2024,
  CAST(`2023 Penetration %` AS DOUBLE) AS penetration_2023,
  CAST(`3 Month Trend` AS DOUBLE) AS trend_3_month,
  CAST(`6 Month Trend` AS DOUBLE) AS trend_6_month,
  CAST(`9 Month Trend` AS DOUBLE) AS trend_9_month,
  CAST(`CY Market Exposure %` AS DOUBLE) AS cy_market_exposure,
  CAST(`PY Market Exposure %` AS DOUBLE) AS py_market_exposure
FROM read_files(
  '/Volumes/{catalog}/{schema}/{volume_name}/account_targeting.csv',
  format => 'csv', header => true, inferSchema => true
)
""")
print("account_targeting created")

## Verify Row Counts

In [ ]:
%sql
SELECT 'hcp_procedure_volume' AS table_name, COUNT(*) AS row_count FROM hcp_procedure_volume
UNION ALL
SELECT 'product_upgrades',    COUNT(*) FROM product_upgrades
UNION ALL
SELECT 'account_targeting',   COUNT(*) FROM account_targeting;